In [0]:

'''
salting - append a random integer  within a small range 0 to n-1 called as salt value to each record
        replicate the key in lookup table 
        distribute and process 
        remove salt 
'''

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
    (1, 101, 100),
    (2, 102, 200),
    (3, 999, 300),
    (4, 999, 400),
    (5, 999, 500),
    (6, 999, 600),
    (7, 999, 700),
    (8, 103, 800)
AS t(order_id, customer_id, amount);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
    (101, 'Alice'),
    (102, 'Bob'),
    (103, 'Charlie'),
    (999, 'Big Customer')
AS t(customer_id, customer_name);

In [0]:
%sql
SELECT
    o.order_id,
    o.customer_id,
    o.amount,
    c.customer_name
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id;

number_of_salts=10


In [0]:
# adding salt to the larger table
from pyspark.sql import functions as F

orders = spark.table("orders")

n = 10
orders_salt = orders.withColumn("salt", F.floor(F.rand(seed=42) * n).cast("int"))

display(orders_salt)

In [0]:
salt_values = spark.range(n).withColumnRenamed("id", "salt")

customers=spark.table("customers")

customers_salted = customers.crossJoin(salt_values)

display(customers_salted)

In [0]:
N = 10

orders_salted = orders.withColumn(
    "salt",
    F.when(
        F.col("customer_id") == 999,
        F.floor(F.rand(seed=42) * N).cast("int")
    ).otherwise(F.lit(0))
)

In [0]:
orders_salted.display()

In [0]:
salt_values = spark.range(N).withColumnRenamed("id", "salt")

normal_customers = customers.filter(
    F.col("customer_id") != 999
).withColumn(
    "salt",
    F.lit(0)
)

skewed_customers = customers.filter(
    F.col("customer_id") == 999
).crossJoin(salt_values)

customers_salted = normal_customers.unionByName(
    skewed_customers
)

In [0]:
orders = spark.range(0, 1000000).select(
    F.col("id").alias("order_id"),
    F.when(
        F.col("id") < 900000,
        F.lit(999)
    ).otherwise(
        (F.col("id") % 10000)
    ).cast("int").alias("customer_id")
)

In [0]:
orders.groupBy("customer_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)

In [0]:
result = orders.join(
    customers,
    "customer_id"
)

In [0]:
result.explain("formatted")